# Inference Speed & Efficiency Benchmarking

In [1]:
import csv
import os
import random
import sys
import time
from types import ModuleType
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

if "torch.distributed.rpc" not in sys.modules:
    try:
        import torch.distributed.rpc
    except RuntimeError:
        rpc_dummy = ModuleType("torch.distributed.rpc")
        rpc_dummy.is_available = lambda: False
        sys.modules["torch.distributed.rpc"] = rpc_dummy

try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "timm"])
    import timm
from skimage.filters import threshold_otsu
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning, module="timm.layers.interpolate")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def resolve_path(candidates, default):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return default

DATA_DIR = resolve_path([os.getenv("DATA_DIR", ""), "./data", "./dataset", "/kaggle/input/lulc-chennai", "../data"], "./data")
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)
NUM_CLASSES = 8
CKPT_NAME = "best_model_final_5.pth"
CVA_WATER_NDWI_THRESHOLD = 0.35
DISPLAY_CONFIDENCE_TAU = 0.60

CONFIG = {
    "img_size": 128,
    "in_chans": 9,
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 5.0e-5,
    "weight_decay": 6.358359e-4,
    "focal_gamma": 0.6136,
    "backbone_lr_ratio": 0.2719,
    "epochs": 160,
    "freeze_epochs": 15,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "label_smoothing": 0.1,
    "class_weight_cap": 5.0,
    "change_pos_weight_cap": 8.0,
    "change_bce_weight": 0.70,
    "change_dice_weight": 0.30,
    "change_task_weight": 1.0,
    "backbone": "swin_tiny_patch4_window7_224",
    "attn_heads": 8,
    "cross_attn_heads": 8,
    "num_layers": 3,
}

print(f"Device: {device}")

Device: cuda


## Spectral Processing

In [2]:
def to_nine_band(patch_5band, eps=1e-6):
    b2, b3, b4, b8, b11 = patch_5band[:5]
    ndvi = (b8 - b4) / (b8 + b4 + eps)
    ndwi = (b3 - b8) / (b3 + b8 + eps)
    ndbi = (b11 - b8) / (b11 + b8 + eps)
    mndwi = (b3 - b11) / (b3 + b11 + eps)
    return np.stack([b2, b3, b4, b8, b11, ndvi, ndwi, ndbi, mndwi], axis=0).astype(np.float32)


def normalize_bands(bands):
    out = bands.copy()
    out[:5] = np.clip(out[:5], 0, 1)
    out[5:] = (np.clip(out[5:], -1, 1) + 1) / 2
    return out

## Model Architecture

In [3]:
class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),         nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=[1,2,4]):
        super().__init__()
        self.stages = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))])
        for r in rates:
            self.stages.append(nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True)))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch*(len(rates)+2),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self, x):
        feats = [s(x) for s in self.stages]
        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.project(torch.cat(feats, dim=1))

def widen_patch_embed(backbone, in_chans):
    proj = backbone.patch_embed.proj; old_w = proj.weight.data
    new_proj = nn.Conv2d(in_chans, proj.out_channels, proj.kernel_size, proj.stride, proj.padding,
                         bias=proj.bias is not None)
    with torch.no_grad():
        new_proj.weight[:,:3] = old_w
        if in_chans > 3:
            new_proj.weight[:,3:] = old_w.mean(1,keepdim=True).repeat(1,in_chans-3,1,1)
        if proj.bias is not None: new_proj.bias[:] = proj.bias.data
    backbone.patch_embed.proj = new_proj

def to_nchw(feat, chs):
    if feat.dim()==4 and feat.shape[1]!=feat.shape[-1] and feat.shape[-1] in chs:
        feat = feat.permute(0,3,1,2).contiguous()
    return feat

class BSALayer(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x):
        h=self.n1(x); x=x+self.attn(h,h,h,need_weights=False)[0]
        h=self.n2(x); return x+self.mlp(h)

class BottleneckSelfAttention(nn.Module):
    def __init__(self, dim, heads=8, num_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([BSALayer(dim,heads) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(dim)
    def forward(self, x):
        B,C,H,W = x.shape
        t = x.permute(0,2,3,1).reshape(B,H*W,C)
        for l in self.layers: t=l(t)
        return self.norm(t).reshape(B,H,W,C).permute(0,3,1,2)

class CrossTransformerBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.na=nn.LayerNorm(dim); self.nb=nn.LayerNorm(dim)
        self.attn12=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.attn21=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.nc=nn.LayerNorm(dim); self.nd=nn.LayerNorm(dim)
        self.proj=nn.Linear(dim*2,dim)
    def forward(self, f1, f2):
        B,C,H,W = f1.shape
        t1=f1.permute(0,2,3,1).reshape(B,H*W,C); t2=f2.permute(0,2,3,1).reshape(B,H*W,C)
        n1,n2=self.na(t1),self.nb(t2)
        c1=t1+self.attn12(n1,n2,n2,need_weights=False)[0]
        c2=t2+self.attn21(n2,n1,n1,need_weights=False)[0]
        out=self.proj(torch.cat([self.nc(c1),self.nd(c2)],dim=-1))
        return out.reshape(B,H,W,C).permute(0,3,1,2)

class SiameseSwinRefined(nn.Module):
    def __init__(self, num_classes=8, in_chans=9, img_size=128,
                 backbone_name="swin_tiny_patch4_window7_224",
                 pretrained=True, attn_heads=8, cross_heads=8, num_layers=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                            features_only=True, img_size=img_size, in_chans=3)
        widen_patch_embed(self.backbone, in_chans)
        self.chs = self.backbone.feature_info.channels()

        self.self_attn   = BottleneckSelfAttention(self.chs[3], attn_heads, num_layers)
        self.aspp        = ASPP(self.chs[3], self.chs[3], rates=[1,2,4])
        self.cross_attn  = CrossTransformerBlock(self.chs[3], cross_heads)

        self.dec3 = UNetDecoderBlock(self.chs[3], self.chs[2], self.chs[2])
        self.dec2 = UNetDecoderBlock(self.chs[2], self.chs[1], self.chs[1])
        self.dec1 = UNetDecoderBlock(self.chs[1], self.chs[0], self.chs[0])

        self.lulc_head = nn.Sequential(
            nn.Conv2d(self.chs[0], self.chs[0]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[0]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[0]//2, num_classes, 1))

        self.change_head = nn.Sequential(
            nn.Conv2d(self.chs[3], self.chs[3]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[3]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[3]//2, 1, 1))

        self.img_size = img_size

    def encode(self, x):
        return [to_nchw(f, self.chs) for f in self.backbone(x)]

    def forward(self, x1, x2):
        f1, f2 = self.encode(x1), self.encode(x2)
        f1a, f2a = self.self_attn(f1[3]), self.self_attn(f2[3])
        f1p, f2p = self.aspp(f1a), self.aspp(f2a)

        s1,s2,s3,_ = f2
        d3 = self.dec3(f2p, s3); d2 = self.dec2(d3,s2); d1 = self.dec1(d2,s1)
        lulc = F.interpolate(self.lulc_head(d1),(self.img_size,self.img_size),mode="bilinear",align_corners=False)

        fused = self.cross_attn(f1p, f2p)
        change = F.interpolate(self.change_head(fused),(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        return lulc, change.squeeze(1)

## Load Checkpoint

In [4]:
model = SiameseSwinRefined(
    num_classes=NUM_CLASSES, in_chans=CONFIG["in_chans"], img_size=CONFIG["img_size"],
    backbone_name=CONFIG["backbone"], pretrained=False,
    attn_heads=CONFIG["attn_heads"], cross_heads=CONFIG["cross_attn_heads"],
    num_layers=CONFIG["num_layers"]
).to(device)

ckpt_path = os.path.join("/kaggle/input/datasets/user/lulc-model-1", CKPT_NAME)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")


Loaded checkpoint from epoch 158


## Parameter Count Breakdown

In [5]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

def count_trainable(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

total = count_params(model)
backbone = count_params(model.backbone)
bsa = count_params(model.self_attn)
aspp = count_params(model.aspp)
cross_attn = count_params(model.cross_attn)
decoder = count_params(model.dec3) + count_params(model.dec2) + count_params(model.dec1)
lulc_head = count_params(model.lulc_head)
change_head = count_params(model.change_head)

components = [
    ("Swin-T Backbone", backbone),
    ("Bottleneck Self-Attention (BSA)", bsa),
    ("ASPP Module", aspp),
    ("Cross-Transformer", cross_attn),
    ("U-Net Decoder (3 blocks)", decoder),
    ("LULC Classification Head", lulc_head),
    ("Change Detection Head", change_head),
]

print(f"{'Component':>35s} | {'Parameters':>12s} | {'%':>6s}")
print("-" * 60)
for name, count in components:
    print(f"{name:>35s} | {count:>12,d} | {count/total*100:>5.1f}%")
print("-" * 60)
print(f"{'TOTAL':>35s} | {total:>12,d} | 100.0%")
print(f"\nTotal: {total/1e6:.2f}M parameters")

                          Component |   Parameters |      %
------------------------------------------------------------
                    Swin-T Backbone |   27,521,274 |  32.7%
    Bottleneck Self-Attention (BSA) |   21,265,152 |  25.2%
                        ASPP Module |   20,063,232 |  23.8%
                  Cross-Transformer |    5,911,296 |   7.0%
           U-Net Decoder (3 blocks) |    6,778,464 |   8.0%
           LULC Classification Head |       42,008 |   0.0%
              Change Detection Head |    2,655,745 |   3.2%
------------------------------------------------------------
                              TOTAL |   84,237,171 | 100.0%

Total: 84.24M parameters


## Inference Latency Benchmark

In [6]:
dummy_x1 = torch.randn(1, 9, 128, 128, device=device)
dummy_x2 = torch.randn(1, 9, 128, 128, device=device)

for _ in range(10):
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        model(dummy_x1, dummy_x2)
if device.type == "cuda":
    torch.cuda.synchronize()

N_RUNS = 100
latencies = []
for _ in range(N_RUNS):
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        model(dummy_x1, dummy_x2)
    if device.type == "cuda":
        torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f"Inference Latency (single patch, batch=1):")
print(f"  Mean:   {latencies.mean():.2f} ms")
print(f"  Std:    {latencies.std():.2f} ms")
print(f"  Median: {np.median(latencies):.2f} ms")
print(f"  Min:    {latencies.min():.2f} ms")
print(f"  Max:    {latencies.max():.2f} ms")
print(f"\nThroughput: {1000/latencies.mean():.1f} patches/sec")

PIXEL_SIZE_M = 10
PATCH_PIXELS = 128
km2_per_patch = (PATCH_PIXELS * PIXEL_SIZE_M / 1000) ** 2
km2_per_min = (1000 / latencies.mean()) * 60 * km2_per_patch
print(f"Coverage rate: {km2_per_min:.1f} km²/min")

Inference Latency (single patch, batch=1):
  Mean:   36.30 ms
  Std:    0.90 ms
  Median: 36.02 ms
  Min:    35.27 ms
  Max:    40.36 ms

Throughput: 27.5 patches/sec
Coverage rate: 2708.1 km²/min


## Batch Throughput

In [7]:
batch_sizes = [1, 2, 4, 8, 16, 32]
throughputs = []

for bs in batch_sizes:
    try:
        dummy_x1 = torch.randn(bs, 9, 128, 128, device=device)
        dummy_x2 = torch.randn(bs, 9, 128, 128, device=device)

        for _ in range(5):
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                model(dummy_x1, dummy_x2)

        if device.type == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(20):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                model(dummy_x1, dummy_x2)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)

        mean_time = np.mean(times)
        tp = bs / mean_time
        throughputs.append(tp)
        print(f"  Batch={bs:>2d} | {mean_time*1000:.1f} ms/batch | {tp:.1f} patches/sec")
    except RuntimeError as e:
        print(f"  Batch={bs:>2d} | OOM")
        throughputs.append(0)
        break

fig, ax = plt.subplots(figsize=(10, 6))
valid = [(bs, tp) for bs, tp in zip(batch_sizes[:len(throughputs)], throughputs) if tp > 0]
if valid:
    ax.plot([v[0] for v in valid], [v[1] for v in valid], "o-", color="#2196F3", linewidth=2, markersize=8)
    ax.set_xlabel("Batch Size", fontsize=13)
    ax.set_ylabel("Throughput (patches/sec)", fontsize=13)
    ax.set_title("Inference Throughput vs Batch Size", fontsize=15, fontweight="bold")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "throughput_vs_batch.png"), dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved throughput_vs_batch.png")

  Batch= 1 | 37.9 ms/batch | 26.4 patches/sec
  Batch= 2 | 39.5 ms/batch | 50.7 patches/sec
  Batch= 4 | 38.3 ms/batch | 104.5 patches/sec
  Batch= 8 | 36.2 ms/batch | 221.2 patches/sec
  Batch=16 | 41.4 ms/batch | 386.6 patches/sec
  Batch=32 | 76.7 ms/batch | 417.4 patches/sec
Saved throughput_vs_batch.png


## GPU Memory Usage

In [8]:
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    dummy_x1 = torch.randn(1, 9, 128, 128, device=device)
    dummy_x2 = torch.randn(1, 9, 128, 128, device=device)
    with torch.no_grad(), torch.amp.autocast("cuda"):
        model(dummy_x1, dummy_x2)
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2
    print(f"GPU Peak Memory (batch=1): {peak_mem:.1f} MB")

    torch.cuda.reset_peak_memory_stats()
    dummy_x1 = torch.randn(32, 9, 128, 128, device=device)
    dummy_x2 = torch.randn(32, 9, 128, 128, device=device)
    with torch.no_grad(), torch.amp.autocast("cuda"):
        model(dummy_x1, dummy_x2)
    peak_mem_32 = torch.cuda.max_memory_allocated() / 1024**2
    print(f"GPU Peak Memory (batch=32): {peak_mem_32:.1f} MB")

    model_mem = sum(p.nelement() * p.element_size() for p in model.parameters()) / 1024**2
    print(f"Model Weight Memory: {model_mem:.1f} MB")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available, skipping GPU memory measurement")

GPU Peak Memory (batch=1): 841.6 MB
GPU Peak Memory (batch=32): 965.2 MB
Model Weight Memory: 321.3 MB
GPU: Tesla T4


## Comparison Table

In [9]:
total_params = sum(p.numel() for p in model.parameters()) / 1e6

print("\nBenchmark Comparison Table:")
print(f"{'Model':>30s} | {'Params (M)':>12s} | {'Type':>20s}")
print("-" * 70)
comparisons = [
    ("U-Net (ResNet-50)", "~25M", "CNN Encoder-Decoder"),
    ("DeepLabV3+ (ResNet-101)", "~59M", "CNN + ASPP"),
    ("Swin-UNet", "~27M", "Transformer U-Net"),
    ("ChangeFormer", "~41M", "Transformer Siamese"),
    (f"Ours (Siamese Swin-T)", f"{total_params:.2f}M", "Hybrid Transformer"),
]
for name, params, arch_type in comparisons:
    print(f"{name:>30s} | {params:>12s} | {arch_type:>20s}")


Benchmark Comparison Table:
                         Model |   Params (M) |                 Type
----------------------------------------------------------------------
             U-Net (ResNet-50) |         ~25M |  CNN Encoder-Decoder
       DeepLabV3+ (ResNet-101) |         ~59M |           CNN + ASPP
                     Swin-UNet |         ~27M |    Transformer U-Net
                  ChangeFormer |         ~41M |  Transformer Siamese
         Ours (Siamese Swin-T) |       84.24M |   Hybrid Transformer
